# DOM207 MP1 — Statistical Analysis, Hypothesis Testing & Business Recommendations

**Dataset:** `Restaurant_Cleaned.csv` (357 rows after cleaning)
**Purpose:** Run the formal hypothesis tests for the report's *Analysis* section, then translate the statistically supported findings into the *Discussion and Recommendations* section — this is a merged/reconciled version of the two draft analysis notebooks, re-verified end-to-end against the actual cleaned file.

### Questions tested
- **HT1 — Lunch vs Dinner:** does mean bill amount differ by meal period?
- **HT2 — Smoker vs Non-smoker:** does mean bill amount differ by smoking status?
- **HT3 — Party size vs Bill amount:** is there a monotonic relationship?
- **HT4 — Bill amount vs Tip:** is there a relationship?
- **HT5 — Day × Time:** are day of week and meal period associated?
- **HT6 (bonus) — Amount across Days:** does mean bill differ across the 4 days?
- **Bonus — Tip% by waiter gender:** does tip percentage differ by the waiter's gender?

### Methodology note
Rather than picking one test type up front, every comparison below first checks its assumptions (**Shapiro-Wilk** for normality per group, **Levene's test** for equal variance). Because the financial variables (`amount`, `tip`) turn out to be strongly right-skewed in this dataset, the **non-parametric test is treated as the primary decision** for every group comparison, with the classical parametric test (Welch's t-test / Pearson) reported alongside for comparison and because it's informative for magnitude/CI purposes. This dual-reporting approach also means the two independent draft analyses (which used different test choices) can be reconciled directly against each other in one place.

Significance level for all tests: **α = 0.05**.


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import itertools

pd.set_option("display.precision", 4)
ALPHA = 0.05

def fmt_p(p):
    return "< 0.0001" if p < 0.0001 else f"{p:.4f}"


## 1. Load data

Reloaded independently so this notebook runs standalone; derived columns (`tip_pct`, `spend_per_person`) and day ordering match the EDA notebook.

In [2]:
df = pd.read_csv("Restaurant_Cleaned.csv")

day_order = ["thursday", "friday", "saturday", "sunday"]
df["day"] = pd.Categorical(df["day"].str.lower(), categories=day_order, ordered=True)
df["smoker"] = df["smoker"].astype(bool)
df["tip_pct"] = (df["tip"] / df["amount"]) * 100
df["spend_per_person"] = df["amount"] / df["partysize"]

print("Shape:", df.shape)
display(df.head())

display(pd.DataFrame({
    "Metric": ["Total observations", "Mean bill", "Median bill", "Mean tip"],
    "Value": [len(df), df["amount"].mean(), df["amount"].median(), df["tip"].mean()],
}))


Shape: (357, 9)


,amount,tip,gender,smoker,day,time,partysize,tip_pct,spend_per_person
0,16.99,1.01,female,False,sunday,dinner,2,5.9447,8.4950
1,10.34,1.66,male,False,sunday,dinner,3,16.0542,3.4467
2,21.01,3.50,male,False,sunday,dinner,3,16.6587,7.0033
3,23.68,3.31,male,False,sunday,dinner,2,13.9780,11.8400
4,24.59,3.61,female,False,sunday,dinner,4,14.6808,6.1475


,Metric,Value
0,Total observations,357.0000
1,Mean bill,21.3264
2,Median bill,19.1000
3,Mean tip,2.9210


## 2. Assumption-checking helper

Runs Shapiro-Wilk per group and Levene's test across groups, so the choice between a parametric and non-parametric test is justified by the data rather than assumed.

In [3]:
def check_assumptions(groups: dict, label: str, verbose=True):
    """groups: {name: pd.Series}. Returns (all_normal, equal_var)."""
    if verbose:
        print(f"--- Assumption check: {label} ---")
    normal_flags = []
    for name, vals in groups.items():
        stat, p = stats.shapiro(vals)
        normal = p > ALPHA
        normal_flags.append(normal)
        if verbose:
            print(f"  Shapiro-Wilk [{name}] (n={len(vals)}): W={stat:.4f}, p={fmt_p(p)} -> {'normal' if normal else 'NOT normal'}")
    lev_stat, lev_p = stats.levene(*groups.values())
    equal_var = lev_p > ALPHA
    if verbose:
        print(f"  Levene's test (equal variance): stat={lev_stat:.4f}, p={fmt_p(lev_p)} -> "
              f"{'equal variance' if equal_var else 'UNEQUAL variance'}")
    return all(normal_flags), equal_var

def welch_ci(a, b, confidence=0.95):
    """CI for (mean(a) - mean(b)) using Welch-Satterthwaite df."""
    a, b = np.asarray(a), np.asarray(b)
    na, nb = len(a), len(b)
    ma, mb = a.mean(), b.mean()
    va, vb = a.var(ddof=1), b.var(ddof=1)
    diff = ma - mb
    se = np.sqrt(va / na + vb / nb)
    df_w = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
    crit = stats.t.ppf((1 + confidence) / 2, df_w)
    return diff, diff - crit * se, diff + crit * se

def rank_biserial(u_stat, n1, n2):
    """Effect size for Mann-Whitney U; ranges -1..1."""
    return 1 - (2 * u_stat) / (n1 * n2)

def cohend(a, b):
    n1, n2 = len(a), len(b)
    pooled_sd = np.sqrt(((n1 - 1) * a.std(ddof=1) ** 2 + (n2 - 1) * b.std(ddof=1) ** 2) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / pooled_sd

def group_desc(groups: dict, val_label="Bill"):
    return pd.DataFrame({
        "Group": list(groups.keys()),
        "N": [len(v) for v in groups.values()],
        f"Mean {val_label}": [v.mean() for v in groups.values()],
        f"Median {val_label}": [v.median() for v in groups.values()],
        "Std Dev": [v.std() for v in groups.values()],
    })

results = []  # summary rows for the final table


## HT1 — Lunch vs Dinner (bill amount)

**H0:** μ(amount | lunch) = μ(amount | dinner)   **H1:** μ(amount | lunch) ≠ μ(amount | dinner)

Business relevance: staffing and pricing decisions by meal period.

In [4]:
lunch = df.loc[df.time == "lunch", "amount"]
dinner = df.loc[df.time == "dinner", "amount"]

display(group_desc({"Lunch": lunch, "Dinner": dinner}, "Bill"))

all_normal, equal_var = check_assumptions({"lunch": lunch, "dinner": dinner}, "amount by time")

# Parametric (Welch) — reported regardless, for magnitude/CI, but NOT the deciding test since data is non-normal
t_stat, t_p = stats.ttest_ind(lunch, dinner, equal_var=False)
diff, ci_lo, ci_hi = welch_ci(lunch, dinner)
print(f"\n[Reference] Welch t-test: t={t_stat:.4f}, p={fmt_p(t_p)}, "
      f"mean diff (lunch-dinner)={diff:.2f}, 95% CI=({ci_lo:.2f}, {ci_hi:.2f})")

# Non-parametric — primary decision, since Shapiro rejected normality for both groups
u_stat, u_p = stats.mannwhitneyu(lunch, dinner, alternative="two-sided")
effect = rank_biserial(u_stat, len(lunch), len(dinner))
decision = "Reject H0" if u_p < ALPHA else "Fail to reject H0"
print(f"[PRIMARY] Mann-Whitney U: U={u_stat:.1f}, p={fmt_p(u_p)}, rank-biserial r={effect:.3f}")
print(f"Decision (alpha={ALPHA}): {decision}")
agree = "agrees with" if (t_p < ALPHA) == (u_p < ALPHA) else "DISAGREES with"
print(f"Note: Welch t-test {agree} the Mann-Whitney decision.")

results.append(dict(test="HT1: Amount ~ Time", method="Mann-Whitney U (primary)", statistic=u_stat,
                     p_value=u_p, effect_size=effect, effect_label="rank-biserial r", decision=decision,
                     reference=f"Welch t p={fmt_p(t_p)}"))


,Group,N,Mean Bill,Median Bill,Std Dev
0,Lunch,132,20.2230,18.745,8.7239
1,Dinner,225,21.9738,19.650,12.0634


--- Assumption check: amount by time ---
  Shapiro-Wilk [lunch] (n=132): W=0.9403, p=< 0.0001 -> NOT normal
  Shapiro-Wilk [dinner] (n=225): W=0.7441, p=< 0.0001 -> NOT normal
  Levene's test (equal variance): stat=0.4076, p=0.5236 -> equal variance

[Reference] Welch t-test: t=-1.5829, p=0.1144, mean diff (lunch-dinner)=-1.75, 95% CI=(-3.93, 0.42)
[PRIMARY] Mann-Whitney U: U=13659.5, p=0.2062, rank-biserial r=0.080
Decision (alpha=0.05): Fail to reject H0
Note: Welch t-test agrees with the Mann-Whitney decision.


### HT1 interpretation
Dinner's sample mean ($21.97) is higher than lunch's ($20.22), but neither test finds this difference statistically significant at α=0.05, and the Welch 95% CI for the mean difference includes zero. **Business implication:** don't justify a dinner-only pricing/expansion push on this comparison alone.

## HT2 — Smoker vs Non-smoker (bill amount)

**H0:** μ(amount | smoker) = μ(amount | non-smoker)   **H1:** μ(amount | smoker) ≠ μ(amount | non-smoker)

In [5]:
smoker_amt = df.loc[df.smoker == True, "amount"]
nonsmoker_amt = df.loc[df.smoker == False, "amount"]

display(group_desc({"Smoker": smoker_amt, "Non-Smoker": nonsmoker_amt}, "Bill"))

all_normal, equal_var = check_assumptions({"smoker": smoker_amt, "non-smoker": nonsmoker_amt}, "amount by smoker")

t_stat, t_p = stats.ttest_ind(smoker_amt, nonsmoker_amt, equal_var=False)
diff, ci_lo, ci_hi = welch_ci(smoker_amt, nonsmoker_amt)
print(f"\n[Reference] Welch t-test: t={t_stat:.4f}, p={fmt_p(t_p)}, "
      f"mean diff (smoker-non-smoker)={diff:.2f}, 95% CI=({ci_lo:.2f}, {ci_hi:.2f})")

u_stat, u_p = stats.mannwhitneyu(smoker_amt, nonsmoker_amt, alternative="two-sided")
effect = rank_biserial(u_stat, len(smoker_amt), len(nonsmoker_amt))
decision = "Reject H0" if u_p < ALPHA else "Fail to reject H0"
print(f"[PRIMARY] Mann-Whitney U: U={u_stat:.1f}, p={fmt_p(u_p)}, rank-biserial r={effect:.3f}")
print(f"Decision (alpha={ALPHA}): {decision}")
agree = "agrees with" if (t_p < ALPHA) == (u_p < ALPHA) else "DISAGREES with"
print(f"Note: Welch t-test {agree} the Mann-Whitney decision (p={fmt_p(t_p)} vs p={fmt_p(u_p)} — "
      f"they can diverge like this when variances differ a lot and the data are skewed, which is why the "
      f"assumption checks above matter for picking which one to trust).")

results.append(dict(test="HT2: Amount ~ Smoker", method="Mann-Whitney U (primary)", statistic=u_stat,
                     p_value=u_p, effect_size=effect, effect_label="rank-biserial r", decision=decision,
                     reference=f"Welch t p={fmt_p(t_p)}"))


,Group,N,Mean Bill,Median Bill,Std Dev
0,Smoker,143,22.4212,18.98,13.9673
1,Non-Smoker,214,20.5949,19.11,8.3455


--- Assumption check: amount by smoker ---
  Shapiro-Wilk [smoker] (n=143): W=0.7210, p=< 0.0001 -> NOT normal
  Shapiro-Wilk [non-smoker] (n=214): W=0.9500, p=< 0.0001 -> NOT normal
  Levene's test (equal variance): stat=5.4557, p=0.0201 -> UNEQUAL variance

[Reference] Welch t-test: t=1.4050, p=0.1615, mean diff (smoker-non-smoker)=1.83, 95% CI=(-0.74, 4.39)
[PRIMARY] Mann-Whitney U: U=15620.0, p=0.7389, rank-biserial r=-0.021
Decision (alpha=0.05): Fail to reject H0
Note: Welch t-test agrees with the Mann-Whitney decision (p=0.1615 vs p=0.7389 — they can diverge like this when variances differ a lot and the data are skewed, which is why the assumption checks above matter for picking which one to trust).


### HT2 interpretation
Levene's test shows smoker/non-smoker variances are significantly different, and neither group is normally distributed — exactly the conditions where a mean-based test (Welch) and a rank-based test (Mann-Whitney) can disagree noticeably. Here the Mann-Whitney result (rank-biserial r ≈ 0, p ≈ 0.74) is the more trustworthy read given the skew, and it does **not** support a bill-amount difference by smoking status. **Business implication:** don't segment pricing/marketing by smoking status on this basis alone.

## HT3 — Party size vs Bill amount

**H0:** ρ(partysize, amount) = 0   **H1:** ρ(partysize, amount) ≠ 0

Party size is discrete and the relationship need not be linear, so Spearman is primary; Pearson is reported too.

In [6]:
pearson_r3, pearson_p3 = stats.pearsonr(df["partysize"], df["amount"])
spearman_r3, spearman_p3 = stats.spearmanr(df["partysize"], df["amount"])

display(pd.DataFrame({"Method": ["Pearson", "Spearman (primary)"],
                       "Correlation": [pearson_r3, spearman_r3],
                       "p-value": [fmt_p(pearson_p3), fmt_p(spearman_p3)]}))

decision = "Reject H0" if spearman_p3 < ALPHA else "Fail to reject H0"
print(f"Decision (alpha={ALPHA}, based on Spearman): {decision}")
if abs(spearman_r3 - pearson_r3) > 0.15:
    print("Note: Pearson (linear) is notably weaker than Spearman (monotonic) here -> the relationship is "
          "real but non-linear, not merely 'weak'. Worth showing a scatterplot with a trend line in the report.")

results.append(dict(test="HT3: Partysize vs Amount", method="Spearman (primary)", statistic=spearman_r3,
                     p_value=spearman_p3, effect_size=spearman_r3, effect_label="rho", decision=decision,
                     reference=f"Pearson r={pearson_r3:.3f}"))


,Method,Correlation,p-value
0,Pearson,0.1223,0.0208
1,Spearman (primary),0.5344,< 0.0001


Decision (alpha=0.05, based on Spearman): Reject H0
Note: Pearson (linear) is notably weaker than Spearman (monotonic) here -> the relationship is real but non-linear, not merely 'weak'. Worth showing a scatterplot with a trend line in the report.


### HT3 interpretation
Larger parties are associated with larger total bills (this is an association, not a causal claim). **Business implication:** factor expected party-size mix into capacity/seating and revenue planning.

## HT4 — Bill amount vs Tip

**H0:** ρ(amount, tip) = 0   **H1:** ρ(amount, tip) ≠ 0

In [7]:
pearson_r4, pearson_p4 = stats.pearsonr(df["amount"], df["tip"])
spearman_r4, spearman_p4 = stats.spearmanr(df["amount"], df["tip"])

display(pd.DataFrame({"Method": ["Pearson", "Spearman"],
                       "Correlation": [pearson_r4, spearman_r4],
                       "p-value": [fmt_p(pearson_p4), fmt_p(spearman_p4)]}))

decision = "Reject H0" if spearman_p4 < ALPHA else "Fail to reject H0"
print(f"Decision (alpha={ALPHA}, based on Spearman — amount is non-normal, see HT1/HT2 checks): {decision}")

results.append(dict(test="HT4: Amount vs Tip", method="Spearman (primary)", statistic=spearman_r4,
                     p_value=spearman_p4, effect_size=spearman_r4, effect_label="rho", decision=decision,
                     reference=f"Pearson r={pearson_r4:.3f}"))


,Method,Correlation,p-value
0,Pearson,0.5374,< 0.0001
1,Spearman,0.6636,< 0.0001


Decision (alpha=0.05, based on Spearman — amount is non-normal, see HT1/HT2 checks): Reject H0


### HT4 interpretation
Bill amount and tip have a clear, statistically significant positive association (both Pearson and Spearman agree here — this relationship is close to linear as well as monotonic). **Business implication:** bill volume is a reasonable proxy for absolute tip volume when forecasting service-staff income, but this is correlation, not causation.

## HT5 — Day × Time (Chi-square test of independence)

**H0:** day of week and meal period are independent   **H1:** they are associated

Business relevance: staffing, table allocation, and inventory planning by shift.

In [8]:
contingency = pd.crosstab(df["day"], df["time"])
print("Observed counts:")
display(contingency)

chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
print("Expected counts under independence:")
display(expected_df.round(2))

n = contingency.sum().sum()
min_dim = min(contingency.shape) - 1
cramers_v = np.sqrt((chi2 / n) / min_dim)

decision = "Reject H0" if chi_p < ALPHA else "Fail to reject H0"
print(f"\nChi-square = {chi2:.4f}, dof = {dof}, p = {fmt_p(chi_p)}")
print(f"Cramer's V (effect size) = {cramers_v:.3f}")
print(f"Decision (alpha={ALPHA}): {decision}")

results.append(dict(test="HT5: Day vs Time", method="Chi-square", statistic=chi2, p_value=chi_p,
                     effect_size=cramers_v, effect_label="Cramer's V", decision=decision, reference="-"))


Observed counts:


time,dinner,lunch
day,,
thursday,19,75
friday,19,20
saturday,97,21
sunday,90,16


Expected counts under independence:


time,dinner,lunch
day,,
thursday,59.24,34.76
friday,24.58,14.42
saturday,74.37,43.63
sunday,66.81,39.19



Chi-square = 117.7616, dof = 3, p = < 0.0001
Cramer's V (effect size) = 0.574
Decision (alpha=0.05): Reject H0


### HT5 interpretation
Day and meal period are strongly associated (Cramer's V ≈ 0.57, a large effect): Thursday skews heavily lunch, while Saturday/Sunday skew heavily dinner. This is an **operational/scheduling pattern**, not evidence that one meal period is inherently more profitable (see HT1). **Business implication:** plan staffing, table allocation, and inventory by day-and-shift rather than treating every shift as identical.

## HT6 (bonus) — Bill amount across all 4 Days

**H0:** mean amount is equal across thursday/friday/saturday/sunday   **H1:** at least one day differs

Not in either draft, but strengthens the Day-related recommendation with a direct amount comparison (rather than only the Day×Time association in HT5).

In [9]:
day_groups = {name: sub["amount"].values for name, sub in df.groupby("day", observed=True)}

all_normal, equal_var = check_assumptions(
    {k: pd.Series(v) for k, v in day_groups.items()}, "amount by day"
)

if all_normal and equal_var:
    stat_val, kw_p = stats.f_oneway(*day_groups.values())
    omnibus_name = "One-way ANOVA"
    eta_sq = None
else:
    stat_val, kw_p = stats.kruskal(*day_groups.values())
    omnibus_name = "Kruskal-Wallis"
    k, n_total = len(day_groups), sum(len(v) for v in day_groups.values())
    eta_sq = (stat_val - k + 1) / (n_total - k)

decision = "Reject H0" if kw_p < ALPHA else "Fail to reject H0"
print(f"\n[PRIMARY] {omnibus_name}: statistic={stat_val:.4f}, p={fmt_p(kw_p)}")
if eta_sq is not None:
    print(f"eta-squared (effect size) = {eta_sq:.3f}")
print(f"Decision (alpha={ALPHA}): {decision}")

results.append(dict(test="HT6 (bonus): Amount ~ Day", method=omnibus_name, statistic=stat_val, p_value=kw_p,
                     effect_size=eta_sq, effect_label="eta-squared", decision=decision, reference="-"))

if kw_p < ALPHA:
    print("\nPost-hoc pairwise Mann-Whitney (Bonferroni-corrected):")
    pairs = list(itertools.combinations(day_groups.keys(), 2))
    bonf_alpha = ALPHA / len(pairs)
    for d1, d2 in pairs:
        u, p = stats.mannwhitneyu(day_groups[d1], day_groups[d2], alternative="two-sided")
        sig = "significant" if p < bonf_alpha else "not significant"
        print(f"  {d1} vs {d2}: U={u:.1f}, p={fmt_p(p)} -> {sig} (adj. alpha={bonf_alpha:.4g})")


--- Assumption check: amount by day ---
  Shapiro-Wilk [thursday] (n=94): W=0.9210, p=< 0.0001 -> NOT normal
  Shapiro-Wilk [friday] (n=39): W=0.9482, p=0.0717 -> normal
  Shapiro-Wilk [saturday] (n=118): W=0.7215, p=< 0.0001 -> NOT normal
  Shapiro-Wilk [sunday] (n=106): W=0.7967, p=< 0.0001 -> NOT normal
  Levene's test (equal variance): stat=1.4996, p=0.2144 -> equal variance

[PRIMARY] Kruskal-Wallis: statistic=12.5938, p=0.0056
eta-squared (effect size) = 0.027
Decision (alpha=0.05): Reject H0

Post-hoc pairwise Mann-Whitney (Bonferroni-corrected):
  thursday vs friday: U=1600.0, p=0.2505 -> not significant (adj. alpha=0.008333)
  thursday vs saturday: U=4428.5, p=0.0118 -> not significant (adj. alpha=0.008333)
  thursday vs sunday: U=3532.0, p=0.0004 -> significant (adj. alpha=0.008333)
  friday vs saturday: U=2174.5, p=0.6087 -> not significant (adj. alpha=0.008333)
  friday vs sunday: U=1800.5, p=0.2356 -> not significant (adj. alpha=0.008333)
  saturday vs sunday: U=5789.0, p=

### HT6 interpretation
The omnibus test finds a significant difference in mean bill amount across days, but the Bonferroni-corrected pairwise comparisons show only **Thursday vs Sunday** survives correction — the effect size (eta-squared ≈ 0.03) is small. **Business implication:** treat this as a mild, secondary signal alongside HT5's much stronger Day×Time scheduling pattern, not as a standalone pricing driver.

## Bonus — Tip percentage by waiter gender

Not required by the assignment, but directly answers a natural staffing/business question the fields support: does tip **percentage** differ by the waiter's gender? (Recall `gender` is the *waiter's* gender per the spec, not the customer's.)

**H0:** μ(tip_pct | male) = μ(tip_pct | female)   **H1:** μ(tip_pct | male) ≠ μ(tip_pct | female)

In [10]:
male_tip = df.loc[df.gender == "male", "tip_pct"]
female_tip = df.loc[df.gender == "female", "tip_pct"]

display(group_desc({"Male waiter": male_tip, "Female waiter": female_tip}, "Tip %"))

all_normal, equal_var = check_assumptions({"male": male_tip, "female": female_tip}, "tip_pct by gender")

t_stat, t_p = stats.ttest_ind(male_tip, female_tip, equal_var=equal_var)
u_stat, u_p = stats.mannwhitneyu(male_tip, female_tip, alternative="two-sided")
effect = rank_biserial(u_stat, len(male_tip), len(female_tip))
decision = "Reject H0" if u_p < ALPHA else "Fail to reject H0"
print(f"[Reference] t-test: p={fmt_p(t_p)}   |   [PRIMARY] Mann-Whitney: U={u_stat:.1f}, p={fmt_p(u_p)}, "
      f"rank-biserial r={effect:.3f}")
print(f"Decision (alpha={ALPHA}): {decision}")

results.append(dict(test="Bonus: Tip% ~ Waiter Gender", method="Mann-Whitney U (primary)", statistic=u_stat,
                     p_value=u_p, effect_size=effect, effect_label="rank-biserial r", decision=decision,
                     reference=f"t-test p={fmt_p(t_p)}"))


,Group,N,Mean Tip %,Median Tip %,Std Dev
0,Male waiter,206,14.6738,14.6663,4.8141
1,Female waiter,151,14.8356,14.6092,5.2244


--- Assumption check: tip_pct by gender ---
  Shapiro-Wilk [male] (n=206): W=0.9906, p=0.2031 -> normal
  Shapiro-Wilk [female] (n=151): W=0.9347, p=< 0.0001 -> NOT normal
  Levene's test (equal variance): stat=0.0217, p=0.8829 -> equal variance
[Reference] t-test: p=0.7625   |   [PRIMARY] Mann-Whitney: U=15545.0, p=0.9938, rank-biserial r=0.001
Decision (alpha=0.05): Fail to reject H0


### Bonus interpretation
No significant difference in tip percentage by waiter gender — staffing/scheduling decisions don't need to account for this.

## Overall Statistical Findings (summary table)

Ready to paste into the report's Analysis section.

In [11]:
summary = pd.DataFrame(results)
summary["p_value"] = summary["p_value"].map(fmt_p)
summary["statistic"] = summary["statistic"].map(lambda x: f"{x:.4f}")
summary["effect_size"] = summary["effect_size"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "-")
summary


,test,method,statistic,p_value,effect_size,effect_label,decision,reference
0,HT1: Amount ~ Time,Mann-Whitney U (primary),13659.5000,0.2062,0.080,rank-biserial r,Fail to reject H0,Welch t p=0.1144
1,HT2: Amount ~ Smoker,Mann-Whitney U (primary),15620.0000,0.7389,-0.021,rank-biserial r,Fail to reject H0,Welch t p=0.1615
2,HT3: Partysize vs Amount,Spearman (primary),0.5344,< 0.0001,0.534,rho,Reject H0,Pearson r=0.122
3,HT4: Amount vs Tip,Spearman (primary),0.6636,< 0.0001,0.664,rho,Reject H0,Pearson r=0.537
4,HT5: Day vs Time,Chi-square,117.7616,< 0.0001,0.574,Cramer's V,Reject H0,-
5,HT6 (bonus): Amount ~ Day,Kruskal-Wallis,12.5938,0.0056,0.027,eta-squared,Reject H0,-
6,Bonus: Tip% ~ Waiter Gender,Mann-Whitney U (primary),15545.0000,0.9938,0.001,rank-biserial r,Fail to reject H0,t-test p=0.7625


## Business Recommendations

The recommendations below are tied directly to the statistically supported findings above — not speculation.

### 1. Use party-size mix in capacity and revenue planning
**Finding (HT3):** party size has a statistically significant positive monotonic relationship with total bill amount (Spearman ρ ≈ 0.53).
**Proposition:** use the expected mix of party sizes as an input for table allocation and revenue forecasting — larger groups reliably generate larger total bills.

### 2. Use bill volume as a proxy for absolute tip volume
**Finding (HT4):** bill amount and tip are strongly, positively associated (Pearson r ≈ 0.54, Spearman ρ ≈ 0.66).
**Proposition:** when forecasting service-staff tip income, bill volume is an informative (associational, not causal) predictor.

### 3. Plan staffing and inventory around the Day × Time pattern, not a blanket lunch/dinner split
**Finding (HT5, HT6):** day of week and meal period are strongly associated (Cramer's V ≈ 0.57) — Thursdays skew lunch-heavy, weekends skew dinner-heavy — and there is a small but real day-to-day difference in average bill (Thursday vs Sunday).
**Proposition:** review staffing, table allocation, and inventory by day-and-shift combination rather than assuming every lunch (or every dinner) shift looks the same.

### 4. Do not base segmentation or pricing decisions on meal period or smoking status alone
**Finding (HT1, HT2):** the lunch-vs-dinner and smoker-vs-non-smoker mean bill differences are **not** statistically significant at α = 0.05 (this holds under both the primary non-parametric test and the reference parametric test).
**Proposition:** the current sample does not justify major targeting, pricing, or expansion decisions built solely around these two comparisons.

### Overall business proposition
**Based on the observed spending patterns, Foodie India should prioritize operational planning around party-size mix and day/meal-period demand patterns — since these show statistically significant, business-meaningful effects — while avoiding pricing or marketing segmentation by meal period or smoking status, where the data do not support a real difference.**


## Limitations

- Findings are based only on the cleaned dataset supplied for this assignment (357 of 365 original rows, after dropping/imputing invalid records — see the preprocessing notebook).
- Statistical significance describes association, not causation.
- Several fields (`tip`, `partysize`, `day`) were partly imputed (KNN / stratified mode) during cleaning; this analysis inherits whatever uncertainty that introduced.
- The Day × Time association and the Thursday/Sunday amount difference describe *this* sample and should be validated against a larger or more recent time window before being used for major operational or expansion decisions.
- All tests use α = 0.05; "fail to reject H0" means the sample doesn't provide sufficient evidence of a difference — it does not prove the groups are identical.
